# XGBoost Fraud Detection Model Training
## Automated Training Pipeline from Supabase

This notebook:
- Loads labeled fraud data from Supabase (`fraud_logs` with review_status = 'approved'/'rejected')
- Trains XGBoost model
- Evaluates performance metrics
- Exports versioned model artifacts
- Logs results to Supabase `model_versions` table

## Step 1: Install and Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import xgboost as xgb
import joblib
import os
from datetime import datetime
import json
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Step 2: Connect to Supabase

In [ ]:
from supabase import create_client
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY must be set in .env file")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print(f"✓ Connected to Supabase: {SUPABASE_URL}")

## Step 3: Load Training Data from Supabase

In [ ]:
print("Loading training data from fraud_logs table...")

# Query only approved/rejected fraud logs (has true labels)
response = supabase.table("fraud_logs") \
    .select("order_id, customer_id, is_fraud, input_features") \
    .in_("review_status", ["approved", "rejected"]) \
    .execute()

fraud_logs = response.data
print(f"✓ Loaded {len(fraud_logs)} labeled samples")

if len(fraud_logs) < 10:
    print("⚠️  WARNING: Less than 10 samples - model training may not be effective")
    print("   Run batch scoring in Phase 3 to generate more fraud_logs first")

# Convert to DataFrame
training_data = []
for log in fraud_logs:
    if log['input_features']:
        row = {**log['input_features'], 'is_fraud': log['is_fraud']}
        training_data.append(row)

df = pd.DataFrame(training_data)

print(f"\n📊 Dataset Info:")
print(f"   Shape: {df.shape}")
print(f"   Fraud cases: {df['is_fraud'].sum()}")
print(f"   Legitimate cases: {(~df['is_fraud']).sum()}")
print(f"   Fraud rate: {df['is_fraud'].mean()*100:.2f}%")
print(f"\n   Column names: {list(df.columns)}")
print(f"\n   First row:")
print(df.head(1))

## Step 4: Data Preprocessing

In [ ]:
print("Preprocessing data...\n")

# Separate features and target
X = df.drop(columns=['is_fraud', 'order_id', 'customer_id'], errors='ignore')
y = df['is_fraud'].astype(int)

# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Create preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop')

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)
X_processed = pd.DataFrame(X_processed, columns=numeric_cols + categorical_cols)

print(f"\n✓ Processed data shape: {X_processed.shape}")

## Step 5: Train-Test Split

In [ ]:
print("Splitting data into train/test sets...\n")

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"  - Fraud cases: {y_train.sum()}")
print(f"  - Legitimate cases: {(~y_train.astype(bool)).sum()}")

print(f"\nTest set: {X_test.shape}")
print(f"  - Fraud cases: {y_test.sum()}")
print(f"  - Legitimate cases: {(~y_test.astype(bool)).sum()}")

## Step 6: Train XGBoost Model

In [ ]:
print("Training XGBoost model...\n")

# Calculate scale_pos_weight to handle class imbalance
scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    verbose=10,
    use_label_encoder=False
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=10,
    verbose=True
)

print("\n✓ Training complete!")

## Step 7: Evaluate Model

In [ ]:
print("Evaluating model performance...\n")

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"Accuracy:   {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:  {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:     {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score:   {f1:.4f}")
print(f"ROC-AUC:    {roc_auc:.4f}")
print("=" * 50)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  TN (True Negatives):  {cm[0,0]}")
print(f"  FP (False Positives): {cm[0,1]}")
print(f"  FN (False Negatives): {cm[1,0]}")
print(f"  TP (True Positives):  {cm[1,1]}")

## Step 8: Save Model Artifacts

In [ ]:
# Create versioned model directory
version_name = f"xgboost_v{datetime.now().strftime('%Y%m%d_%H%M%S')}"
model_dir = f"python/models/{version_name}"

os.makedirs(model_dir, exist_ok=True)

print(f"Saving model artifacts to {model_dir}...\n")

# Save model
joblib.dump(model, f"{model_dir}/xgboost_model.joblib")
print(f"✓ Saved xgboost_model.joblib")

# Save column names
joblib.dump(X_processed.columns.tolist(), f"{model_dir}/train_columns.joblib")
print(f"✓ Saved train_columns.joblib")

# Save imputers
joblib.dump(preprocessor.named_transformers_['num'].named_steps['imputer'],
            f"{model_dir}/num_imputer.joblib")
print(f"✓ Saved num_imputer.joblib")

joblib.dump(preprocessor.named_transformers_['cat'].named_steps['imputer'],
            f"{model_dir}/cat_imputer.joblib")
print(f"✓ Saved cat_imputer.joblib")

# Save scaler
scaler = StandardScaler()
X_numeric = X_processed[[c for c in numeric_cols if c in X_processed.columns]]
if len(X_numeric.columns) > 0:
    scaler.fit(X_numeric)
    joblib.dump(scaler, f"{model_dir}/scaler.joblib")
    print(f"✓ Saved scaler.joblib")

print(f"\n✅ All artifacts saved to {model_dir}/")

## Step 9: Log to Supabase

In [ ]:
print("Logging model version to Supabase...\n")

try:
    response = supabase.table("model_versions").insert({
        "version_name": version_name,
        "model_type": "xgboost",
        "status": "staged",
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "training_samples": len(X_train),
        "fraud_samples": int(y_train.sum()),
        "trained_on_data_until": datetime.now().isoformat(),
        "notes": f"Trained from {len(fraud_logs)} labeled samples. Fraud rate: {df['is_fraud'].mean()*100:.2f}%"
    }).execute()
    
    print(f"✓ Model version logged: {version_name}")
    print(f"\n📋 Summary:")
    print(f"   Version: {version_name}")
    print(f"   Status: staged")
    print(f"   Location: {model_dir}/")
    print(f"\n🎯 Next Steps:")
    print(f"   1. Review metrics above")
    print(f"   2. Go to http://localhost:3000/model-versions")
    print(f"   3. Click 'Deploy' to make it production")
    print(f"   4. All future predictions will use this model")
    
except Exception as e:
    print(f"❌ Error logging to Supabase: {e}")
    print(f"   Model artifacts are saved locally but not registered in Supabase")

## Summary

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nModel Version: {version_name}")
print(f"Saved to: {model_dir}/")
print(f"\nPerformance:")
print(f"  Accuracy:  {accuracy*100:.2f}%")
print(f"  Precision: {precision*100:.2f}%")
print(f"  Recall:    {recall*100:.2f}%")
print(f"  F1-Score:  {f1:.4f}")
print(f"\nTraining Data:")
print(f"  Total Samples: {len(X_train)}")
print(f"  Fraud Cases: {int(y_train.sum())}")
print(f"  Fraud Rate: {df['is_fraud'].mean()*100:.2f}%")
print("\n" + "="*60)